# R6 DINOv3 Cat ReID — ViT-L (warm-started from R5 best params)

Changes from R5:
- Backbone: `vit_base_patch16_dinov3` → `vit_large_patch16_dinov3` (768 → 1024 features)
- No Optuna — R5's tuned hyperparams baked in (transfer well from base to large)
- Random 85/15 split (matches R5)

Deploy target: Modal T4 (ViT-L is ~550MB fp16, fits easily). Train on A100.

In [ ]:
!pip -q install "timm>=1.0.20" pytorch-metric-learning scikit-learn pandas tqdm matplotlib pillow

In [ ]:
import timm
MODEL_NAME = "vit_large_patch16_dinov3"
cands = timm.list_models("*dinov3*")
assert MODEL_NAME in cands, f"{MODEL_NAME} not in timm. Available large variants: {[m for m in cands if 'large' in m]}"
m = timm.create_model(MODEL_NAME, pretrained=True, num_classes=0)
print(f"{MODEL_NAME}: {m.num_features} features, {sum(p.numel() for p in m.parameters())/1e6:.1f}M params")
del m

In [ ]:
import os, json, math, random, zipfile, shutil, time
from collections import Counter
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from tqdm.notebook import tqdm
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from PIL import Image
import timm
from pytorch_metric_learning import losses, samplers
from google.colab import drive
drive.mount("/content/drive")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

SEED = 42
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

DRIVE_PATH = "/content/drive/MyDrive"
RUN_DIR = os.path.join(DRIVE_PATH, "R6_cat_DINOv3")
os.makedirs(RUN_DIR, exist_ok=True)
OUT_BASE = "R6_cat_DINOv3"

device = torch.device("cuda")
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / (1024**3)
print(f"GPU: {props.name}, VRAM: {vram_gb:.1f} GB")

USE_AMP = True
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

# ViT-L is ~3.5x ViT-B compute; downsize batches accordingly.
if vram_gb >= 70:   IMG_SIZE=448; BATCH_SIZE=48; GRAD_ACCUM=2; NUM_WORKERS=12
elif vram_gb >= 35: IMG_SIZE=448; BATCH_SIZE=24; GRAD_ACCUM=4; NUM_WORKERS=8
elif vram_gb >= 14: IMG_SIZE=336; BATCH_SIZE=12; GRAD_ACCUM=8; NUM_WORKERS=4
else:               raise RuntimeError(f"ViT-L needs >=14GB VRAM, got {vram_gb:.1f}")
print(f"IMG={IMG_SIZE} BS={BATCH_SIZE} ACCUM={GRAD_ACCUM} WORKERS={NUM_WORKERS}")

In [ ]:
ZIP_NAME = "R6_TomCat_Training.zip"
ZIP_PATH = os.path.join(DRIVE_PATH, ZIP_NAME)
assert os.path.exists(ZIP_PATH), f"Zip not found at: {ZIP_PATH}"
print(f"Found: {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1e9:.2f} GB)")

RAW_DIR = "/content/raw_data"
if os.path.exists(RAW_DIR): shutil.rmtree(RAW_DIR)
print("Unzipping...")
with zipfile.ZipFile(ZIP_PATH, "r") as zf: zf.extractall(RAW_DIR)

csv_path = img_folder = None
for root, dirs, files in os.walk(RAW_DIR):
    for fn in files:
        if fn.lower().endswith('.csv') and 'pic' in fn.lower():
            csv_path = os.path.join(root, fn)
    for d in dirs:
        if 'totalpics' in d.lower():
            img_folder = os.path.join(root, d)
assert csv_path and img_folder, f"Missing CSV or TotalPicsOfCats. Got: csv={csv_path} img={img_folder}"
print(f"CSV: {csv_path}\nImages: {img_folder}")

ok_ext = {".jpg", ".jpeg", ".png"}
sn_to_path = {Path(fn).stem.lower(): os.path.join(img_folder, fn) for fn in os.listdir(img_folder) if Path(fn).suffix.lower() in ok_ext}
print(f"Indexed {len(sn_to_path)} images")

In [ ]:
from PIL import ImageFile
Image.MAX_IMAGE_PIXELS = 500_000_000   # allow up to ~500MP; one source JPEG (sn11405) is ~199MP
ImageFile.LOAD_TRUNCATED_IMAGES = True  # do not bail on slightly-corrupt JPEGs

CROP_PAD = 0.15; MIN_CROP_PX = 40
df = pd.read_csv(csv_path)
print(f"CSV: {len(df)} rows")

CROP_DIR = "/content/R6_all_crops"
if os.path.exists(CROP_DIR): shutil.rmtree(CROP_DIR)

crop_count = skip_rej = skip_noimg = skip_bad = skip_tiny = 0
for _, row in tqdm(df.iterrows(), total=len(df), desc="Cropping"):
    sn = str(row['Serial Number']).strip().lower()
    box_ids = str(row.get('Box Cat IDs', '')).strip()
    box_coords = str(row.get('Box Coordinates', '')).strip()
    if box_ids in ('Rejected','','nan') or box_coords in ('Rejected','','nan'):
        skip_rej += 1; continue
    if sn not in sn_to_path: skip_noimg += 1; continue
    cats, coords = box_ids.split('|'), box_coords.split('|')
    try:
        img = Image.open(sn_to_path[sn]).convert('RGB'); W, H = img.size
    except: skip_bad += 1; continue
    for cat, coord in zip(cats, coords):
        cat = cat.strip()
        if not cat or cat == 'Rejected': continue
        parts = coord.strip().split()
        if len(parts) != 4: skip_bad += 1; continue
        try: cx,cy,bw,bh = map(float, parts)
        except: skip_bad += 1; continue
        pw, ph = bw*W, bh*H; padw, padh = pw*CROP_PAD, ph*CROP_PAD
        x1 = max(0, int(cx*W - pw/2 - padw)); y1 = max(0, int(cy*H - ph/2 - padh))
        x2 = min(W, int(cx*W + pw/2 + padw)); y2 = min(H, int(cy*H + ph/2 + padh))
        if (x2-x1) < MIN_CROP_PX or (y2-y1) < MIN_CROP_PX: skip_tiny += 1; continue
        cdir = os.path.join(CROP_DIR, cat); os.makedirs(cdir, exist_ok=True)
        img.crop((x1,y1,x2,y2)).save(os.path.join(cdir, f"{sn}_{cat}_{crop_count}.jpg"), quality=95)
        crop_count += 1

print(f"\nCropped: {crop_count}")
print(f"Skipped: rejected={skip_rej} no_image={skip_noimg} bad_box={skip_bad} tiny={skip_tiny}")
cc = {d: len(os.listdir(os.path.join(CROP_DIR,d))) for d in sorted(os.listdir(CROP_DIR)) if os.path.isdir(os.path.join(CROP_DIR,d))}
small = {k:v for k,v in cc.items() if v < 8}
print(f"{len(cc)} classes total; will drop {len(small)} with <8 crops")

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
MIN_SAMPLES = 8; VAL_FRAC = 0.15

all_files, all_labels = [], []
for cn in sorted(os.listdir(CROP_DIR)):
    cd = os.path.join(CROP_DIR, cn)
    if not os.path.isdir(cd): continue
    imgs = [f for f in os.listdir(cd) if Path(f).suffix.lower() in ok_ext]
    if len(imgs) < MIN_SAMPLES: continue
    for fn in imgs: all_files.append(os.path.join(cd, fn)); all_labels.append(cn)
print(f"After MIN_SAMPLES={MIN_SAMPLES} prune: {len(all_files)} crops, {len(set(all_labels))} cats")

sss = StratifiedShuffleSplit(n_splits=1, test_size=VAL_FRAC, random_state=SEED)
train_idx, val_idx = next(sss.split(all_files, all_labels))

SPLIT_DIR = "/content/R6_split"
train_root, val_root = os.path.join(SPLIT_DIR, "train"), os.path.join(SPLIT_DIR, "val")
for d in [train_root, val_root]:
    if os.path.exists(d): shutil.rmtree(d)
    os.makedirs(d)
for idx_set, root in [(train_idx, train_root), (val_idx, val_root)]:
    for i in idx_set:
        dest = os.path.join(root, all_labels[i]); os.makedirs(dest, exist_ok=True)
        shutil.copy2(all_files[i], dest)

common = sorted(set(os.listdir(train_root)) & set(os.listdir(val_root)))
for root in [train_root, val_root]:
    for n in os.listdir(root):
        if n not in common: shutil.rmtree(os.path.join(root, n))
nt = sum(len(os.listdir(os.path.join(train_root,d))) for d in os.listdir(train_root))
nv = sum(len(os.listdir(os.path.join(val_root,d))) for d in os.listdir(val_root))
print(f"Final: {nt} train / {nv} val / {len(common)} cats")
with open(os.path.join(RUN_DIR, f"{OUT_BASE}_classes.json"), "w") as f: json.dump(common, f, indent=2)

In [ ]:
class ReIDModel(nn.Module):
    def __init__(self, model_name, emb_dim, drop_path_rate=0.1):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0, drop_path_rate=drop_path_rate)
        self.head = nn.Sequential(nn.Linear(self.backbone.num_features, emb_dim), nn.BatchNorm1d(emb_dim), nn.PReLU())
    def forward(self, x):
        return F.normalize(self.head(self.backbone(x)), p=2, dim=1)

MEAN=[0.485,0.456,0.406]; STD=[0.229,0.224,0.225]

def make_train_tf(sz, aug=0.3):
    return transforms.Compose([
        transforms.RandomResizedCrop(sz, scale=(max(0.4,0.7-aug),1.0), ratio=(0.8,1.2)),
        transforms.RandomHorizontalFlip(0.5),
        transforms.ColorJitter(brightness=aug, contrast=aug, saturation=aug, hue=min(0.15,aug*0.4)),
        transforms.RandomGrayscale(p=0.05+aug*0.1),
        transforms.GaussianBlur(5, sigma=(0.1,1.0+aug)),
        transforms.RandomPerspective(distortion_scale=aug*0.15, p=0.2),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD),
        transforms.RandomErasing(p=0.15+aug*0.2, scale=(0.02,0.15)),
    ])
def make_light_tf(sz):
    return transforms.Compose([transforms.Resize((sz,sz)), transforms.RandomHorizontalFlip(0.5), transforms.ToTensor(), transforms.Normalize(MEAN,STD)])
def make_val_tf(sz):
    return transforms.Compose([transforms.Resize((sz,sz)), transforms.ToTensor(), transforms.Normalize(MEAN,STD)])

@torch.inference_mode()
def extract_embs(model, loader):
    model.eval(); embs, labs = [], []
    for imgs, y in loader:
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE, enabled=USE_AMP):
            e = model(imgs.to(device, non_blocking=True))
        embs.append(F.normalize(e, p=2, dim=1).cpu()); labs.append(y.cpu())
    return torch.cat(embs), torch.cat(labs)

def full_metrics(qe, ql, ge, gl, map_k=100):
    q, g, qlb, glb = qe.to(device), ge.to(device), ql.to(device), gl.to(device)
    K = min(map_k, g.shape[0]); _, ti = (q @ g.T).topk(K, dim=1, largest=True, sorted=True)
    matches = (glb[ti] == qlb.unsqueeze(1))
    r1 = matches[:,:1].any(1).float().mean().item()
    r5 = matches[:,:5].any(1).float().mean().item()
    nc = int(max(qlb.max(), glb.max())+1); pos = torch.bincount(glb, minlength=nc)[qlb]
    cum = matches.float().cumsum(1); ranks = torch.arange(1,K+1,device=device).float().unsqueeze(0)
    mAP = ((cum/ranks)*matches.float()).sum(1) / pos.float().clamp(min=1).clamp(max=float(K))
    return {"recall@1": r1, "recall@5": r5, f"mAP@{K}": mAP.mean().item()}

print("Ready.")

In [ ]:
# R5 Optuna-tuned hyperparams (transfer well to ViT-L)
LR_BB = 6.63580248520245e-05
LR_HD = 0.0003235188302117385
WD    = 0.02142387495644906
AF_M  = 16.90675050858071
AF_S  = 30.927151442351786
DP    = 0.13129583050668675
EMB   = 512
AUG   = 0.31527522380668227

# Training schedule
EP_F  = 3    # Phase A: frozen backbone
EP_FT = 30   # Phase B: full unfreeze + cosine decay
EP_P  = 5    # Phase C: polish at 0.25x LR + light augs
PAT   = 8    # Phase B early-stopping patience
TOT   = EP_F + EP_FT + EP_P
M_PER_CLASS = 2; EVAL_BS = 128; MAP_K = 100

print(f"Schedule: {EP_F}+{EP_FT}+{EP_P}={TOT} ep | EMB={EMB} | AF(m={AF_M:.1f}, s={AF_S:.1f})")

In [ ]:
set_seed(SEED)
ttf, ltf, vtf = make_train_tf(IMG_SIZE, AUG), make_light_tf(IMG_SIZE), make_val_tf(IMG_SIZE)
ds_tr = datasets.ImageFolder(train_root, transform=ttf)
ds_va = datasets.ImageFolder(val_root, transform=vtf)
ds_gal = datasets.ImageFolder(train_root, transform=vtf)
assert ds_va.class_to_idx == ds_tr.class_to_idx
ncls = len(ds_tr.classes)
with open(os.path.join(RUN_DIR, f"{OUT_BASE}_class_to_idx.json"), "w") as f: json.dump(ds_tr.class_to_idx, f, indent=2)
smp = samplers.MPerClassSampler(ds_tr.targets, m=M_PER_CLASS, batch_size=BATCH_SIZE, length_before_new_iter=len(ds_tr))
dl_t = DataLoader(ds_tr, batch_size=BATCH_SIZE, sampler=smp, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True, prefetch_factor=4)
dl_v = DataLoader(ds_va, batch_size=EVAL_BS, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
dl_g = DataLoader(ds_gal, batch_size=EVAL_BS, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
print(f"Train: {len(ds_tr)} | Val: {len(ds_va)} | Classes: {ncls}")

model = ReIDModel(MODEL_NAME, EMB, drop_path_rate=DP).to(device)
lf = losses.ArcFaceLoss(num_classes=ncls, embedding_size=EMB, margin=AF_M, scale=AF_S).to(device)
op = torch.optim.AdamW([
    {"params":model.backbone.parameters(),"lr":LR_BB},
    {"params":model.head.parameters(),"lr":LR_HD},
    {"params":lf.parameters(),"lr":LR_HD},
], weight_decay=WD)
sc = torch.amp.GradScaler('cuda', enabled=USE_AMP)
spe = int(math.ceil(len(dl_t)/GRAD_ACCUM)); tot_s = TOT*spe; wu_s = int(0.05*tot_s)
blrs = [LR_BB, LR_HD, LR_HD]
def set_lrs(step):
    m = (step+1)/max(1,wu_s) if step<wu_s else 0.5*(1+math.cos(math.pi*min(1,(step-wu_s)/max(1,tot_s-wu_s))))
    for i,pg in enumerate(op.param_groups): pg['lr'] = blrs[i]*m
rc = {"model":MODEL_NAME,"emb":EMB,"img":IMG_SIZE,"bs":BATCH_SIZE,"accum":GRAD_ACCUM,
      "lr_bb":LR_BB,"lr_hd":LR_HD,"wd":WD,"af_m":AF_M,"af_s":AF_S,"dp":DP,"aug":AUG,
      "tot_ep":TOT,"ncls":ncls,"warm_start":"R5_best_params"}
with open(os.path.join(RUN_DIR, f"{OUT_BASE}_config.json"), "w") as f: json.dump(rc, f, indent=2)
print(f"Steps/ep={spe} total={tot_s} warmup={wu_s}")

In [ ]:
def save_ckpt(tag, ei):
    torch.save({"epoch":ei,"gs":gs,"model":model.state_dict(),"lf":lf.state_dict(),
        "op":op.state_dict(),"sc":sc.state_dict(),"cfg":rc,
        "best_r1":best_r1,"best_ep":best_ep,"no_imp":no_imp,"blrs":blrs},
        os.path.join(RUN_DIR, f"{OUT_BASE}_{tag}.pt"))

def ev():
    Eg,Lg = extract_embs(model, dl_g); Eq,Lq = extract_embs(model, dl_v)
    return full_metrics(Eq, Lq, Eg, Lg, map_k=MAP_K)

def train_ep(ei):
    global gs
    model.train(); rl = 0.0; op.zero_grad(set_to_none=True)
    pb = tqdm(dl_t, desc=f"Ep {ei+1}/{TOT}", leave=False)
    for si,(im,lb) in enumerate(pb):
        im,lb = im.to(device,non_blocking=True), lb.to(device,non_blocking=True)
        set_lrs(gs)
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE, enabled=USE_AMP):
            loss = lf(model(im), lb) / GRAD_ACCUM
        sc.scale(loss).backward(); rl += loss.item()*GRAD_ACCUM
        if ((si+1)%GRAD_ACCUM==0) or (si+1==len(dl_t)):
            sc.unscale_(op); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            sc.step(op); sc.update(); op.zero_grad(set_to_none=True); gs += 1
        pb.set_postfix(loss=f"{rl/(si+1):.4f}", lr=f"{op.param_groups[1]['lr']:.2e}")
    return rl / len(dl_t)

gs = 0; SE = 0; ml = []; best_r1 = -1.0; best_ep = -1; no_imp = 0
rp = os.path.join(RUN_DIR, f"{OUT_BASE}_last.pt")
mc = os.path.join(RUN_DIR, f"{OUT_BASE}_metrics.csv")
if os.path.exists(rp):
    print(f"RESUMING from {rp}")
    ck = torch.load(rp, map_location=device, weights_only=False)
    model.load_state_dict(ck["model"]); lf.load_state_dict(ck["lf"])
    op.load_state_dict(ck["op"]); sc.load_state_dict(ck["sc"])
    gs = ck["gs"]; SE = ck["epoch"]+1
    best_r1 = ck.get("best_r1",-1); best_ep = ck.get("best_ep",-1)
    no_imp = ck.get("no_imp",0); blrs = ck.get("blrs",blrs)
    if os.path.exists(mc): ml = pd.read_csv(mc).to_dict('records')
    print(f"  Epoch {SE}, step {gs}, best R@1={best_r1:.4f}")
else: print("Fresh start.")

print(f"\n{'='*50}\nPHASE A: Frozen ({EP_F} ep)\n{'='*50}")
for p in model.backbone.parameters(): p.requires_grad = False
for e in range(EP_F):
    ei = e
    if ei < SE: gs += spe; continue
    a = train_ep(ei); m = ev()
    print(f"  ep {ei+1}: loss={a:.4f} R@1={m['recall@1']:.3f} R@5={m['recall@5']:.3f}")
    ml.append({"epoch":ei+1,"phase":"frozen","loss":a,**m})
    save_ckpt("last",ei); pd.DataFrame(ml).to_csv(mc,index=False)
    if m["recall@1"] > best_r1:
        best_r1, best_ep = m["recall@1"], ei+1; save_ckpt("best",ei)
        torch.save(model.state_dict(), os.path.join(RUN_DIR, f"{OUT_BASE}_encoder.pth"))

print(f"\n{'='*50}\nPHASE B: Finetune ({EP_FT} ep, patience={PAT})\n{'='*50}")
for p in model.backbone.parameters(): p.requires_grad = True
for e in range(EP_FT):
    ei = EP_F + e
    if ei < SE: gs += spe; continue
    a = train_ep(ei); m = ev()
    print(f"  ep {ei+1}: loss={a:.4f} R@1={m['recall@1']:.3f} R@5={m['recall@5']:.3f}")
    ml.append({"epoch":ei+1,"phase":"finetune","loss":a,**m})
    save_ckpt("last",ei); pd.DataFrame(ml).to_csv(mc,index=False)
    if m["recall@1"] > best_r1:
        best_r1, best_ep, no_imp = m["recall@1"], ei+1, 0; save_ckpt("best",ei)
        torch.save(model.state_dict(), os.path.join(RUN_DIR, f"{OUT_BASE}_encoder.pth"))
        print(f"  ** New best R@1={best_r1:.4f} **")
    else: no_imp += 1
    if no_imp >= PAT: print(f"  Early stop at ep {ei+1}"); break

print(f"\n{'='*50}\nPHASE C: Polish ({EP_P} ep, 0.25x LR)\n{'='*50}")
bp = os.path.join(RUN_DIR, f"{OUT_BASE}_best.pt")
if os.path.exists(bp):
    model.load_state_dict(torch.load(bp, map_location=device, weights_only=False)["model"])
    print(f"Loaded best (R@1={best_r1:.4f})")
blrs = [lr*0.25 for lr in blrs]; ds_tr.transform = ltf
for e in range(EP_P):
    ei = EP_F + EP_FT + e
    if ei < SE: gs += spe; continue
    a = train_ep(ei); m = ev()
    print(f"  ep {ei+1}: loss={a:.4f} R@1={m['recall@1']:.3f}")
    ml.append({"epoch":ei+1,"phase":"polish","loss":a,**m})
    save_ckpt("last",ei); pd.DataFrame(ml).to_csv(mc,index=False)
    if m["recall@1"] > best_r1:
        best_r1, best_ep = m["recall@1"], ei+1; save_ckpt("best",ei)
        torch.save(model.state_dict(), os.path.join(RUN_DIR, f"{OUT_BASE}_encoder.pth"))
        print(f"  ** New best R@1={best_r1:.4f} **")
save_ckpt("final",ei); pd.DataFrame(ml).to_csv(mc,index=False)
print(f"\n{'='*60}\nDONE — Best R@1: {best_r1:.4f} at epoch {best_ep}\n{'='*60}")